# Final Project Report: Lightweight Malicious PDF Detector

**Author:** Saed Abdalgani  
**Date:** April 2026  
**Course:** Machine Learning Security  

---

## 1. Executive Summary
This project presents a high-performance, edge-optimized machine learning pipeline for detecting malicious PDF documents. By extracting 37 static structural and metadata features, we trained a PyTorch Multi-Layer Perceptron (MLP) capable of achieving 99.80% accuracy. To meet strict non-functional requirements for deployment speed and footprint, the model was statically quantized to INT8, resulting in a >50% size reduction (down to 550KB) and single-sample inference times under 10ms. Finally, a local Large Language Model (Gemma 4 via Ollama) was integrated to provide explainable threat intelligence on identified suspicious features without compromising document privacy.

---

## 2. Problem Statement & Motivation
PDF files are frequently used as vectors for malware delivery through embedded JavaScript, malicious `/OpenAction` triggers, and obfuscated shellcode. Traditional signature-based detection is easily bypassed by polymorphic malware. 

**The objective is to build a detector that is:**
1. **Behavioral & Statistical:** Catches novel threats by analyzing structural anomalies rather than known hashes.
2. **Secure & Private:** Processes files entirely in-memory with zero outbound network calls.
3. **Explainable:** Gives SOC analysts actionable intelligence using AI, not just a binary "Malicious/Safe" verdict.
4. **Lightweight:** Deployable on commodity hardware (CPU-only) with minimal latency.

---

## 3. Dataset Description & Preprocessing
The dataset used is the **CIC Evasive-PDFMal2022** dataset containing ~10,025 samples of benign and malicious PDFs.

**Key Preprocessing Steps:**
1. **Deduplication:** Removed identical feature vectors.
2. **Missing Value Imputation:** Handled sparse/corrupted rows via median imputation.
3. **Train/Val/Test Split:** Stratified split ensuring 15% holdout test set.
4. **SMOTE Balancing:** Applied Synthetic Minority Over-sampling Technique (SMOTE) strictly on the training set to resolve class imbalances.
5. **Standardization:** Fit a `StandardScaler` on the training distribution to normalize numerical ranges for neural network convergence.

---

## 4. EDA Key Findings
Exploratory Data Analysis revealed several highly discriminative features:

1. **JavaScript Presence:** Malicious files heavily utilize `/JS` and `/JavaScript` tags.
2. **Obfuscation Markers:** The `obfuscation_count` (hex-encoded `#XX` tags) is near-zero in benign files but prevalent in malicious ones.
3. **Auto-Execution:** `/OpenAction` tags strongly correlate with malicious behavior.

![Feature Importance Heatmap](../reports/figures/rf_confusion_matrix.png)

*(Note: Replace the image above with actual EDA distributions if needed)*

---

## 5. Feature Engineering Methodology
The extraction pipeline was custom-built without relying on external cloud APIs. 
- **Structural Features (Regex-based):** Scans raw bytes for 25 distinct PDF keywords (e.g., `/ObjStm`, `/Filter`, `/AA`).
- **Metadata Features (PyPDF2):** Extracts 12 high-level properties like page count, encryption status, and presence of embedded files.
- **Security Boundaries:** A strict 30-second timeout is enforced, and file limits cap at 50MB to prevent ReDoS (Regular Expression Denial of Service) and Out-Of-Memory attacks.

---

## 6. Model Comparison Results
Multiple algorithms were evaluated (Random Forest, XGBoost, LightGBM, and PyTorch MLP). The MLP was selected due to its high accuracy and suitability for PyTorch Post-Training Quantization.

| Model | Accuracy | F1 Score | AUC-ROC |
|-------|----------|----------|---------|
| Random Forest | 99.82% | 0.998 | 0.999 |
| XGBoost | 99.78% | 0.998 | 0.999 |
| LightGBM | 99.80% | 0.998 | 0.999 |
| **MLP (FP32)** | **99.80%** | **0.998** | **0.999** |

![ROC Curves](../reports/figures/roc_curves.png)
![Precision-Recall Curves](../reports/figures/pr_curves.png)

---

## 7. Quantization Analysis
To deploy the application as a lightweight edge service, we applied **Static INT8 Quantization** to the MLP.

- **Size Reduction:** The model footprint was reduced from **1.25 MB to 0.55 MB** (56% reduction).
- **Speedup:** Inference latency dropped from ~12ms to ~6ms on standard CPUs.
- **Accuracy Retention:** The accuracy drop was <0.02%, far below the 1% threshold, confirming that the model effectively compressed its learned representations into 8-bit integers without degrading performance.

---

## 8. LLM Integration & Threat Analysis
To provide explainability, a localized LLM (`Gemma 4`) runs via Ollama. It analyzes the specific features that deviate by $>2\sigma$ from the benign baseline.

**Example Generated Threat Report:**
> **Severity: CRITICAL**
> 
> **Threat Assessment:** The ML classifier flagged this document with 99.7% confidence. The presence of 12 heavily obfuscated streams combined with an auto-executing `/OpenAction` trigger is indicative of a dropper or exploit payload targeting PDF reader vulnerabilities.
> 
> **Attack Vector:** Drive-by download execution triggered upon opening the document.
> 
> **Remediation:** Quarantine immediately. Do not attempt to render the document in any reader. Submit the hash to local threat intelligence platforms.

*Security Constraint: The prompt strictly runs locally. If system RAM is beneath 3GB, the LLM module safely disables itself to prevent OS crashes.*

---

## 9. Streamlit Application
The end-user application was built using Streamlit, featuring a dark-themed, glassmorphic UI.

**Features:**
- Instant Drag-and-Drop scanning.
- Radar charts comparing the uploaded file against benign baselines.
- Interactive Chatbot for asking the AI follow-up questions regarding the threat.

![Dashboard UI](../app/assets/screenshot_dashboard.png)

---

## 10. Conclusions & Future Work

**Conclusions:**
The project successfully demonstrates that complex, evasive malware inside PDFs can be detected securely and instantly using a fusion of static feature engineering and lightweight neural networks. By bringing an LLM to the edge, the system bridges the gap between binary classification and human-readable threat intelligence.

**Future Work:**
1. **Dynamic Analysis Integration:** Coupling the static extractor with a secure sandbox to trace API calls and child processes.
2. **Javascript Deobfuscation:** Implementing AST (Abstract Syntax Tree) parsing to extract and evaluate the actual payload of the embedded JS.
3. **Cross-Platform Compilation:** Porting the INT8 PyTorch model to ONNX Runtime for C++ or Rust deployment.